<https://colab.research.google.com/github/philzook58/philzook58.github.io/blob/master/pynb/asm_verify_demo.ipynb>

# Z3 and SMT

In [ ]:
! uv pip install git+https://github.com/philzook58/knuckledragger.git[pypcode]

In [8]:
from z3 import *
x = BitVec("x", 64)
prove(x < x + 1)

counterexample
[x = 9223372036854775807]


# Add 1 x64

In [9]:
%%file /tmp/simple.s

.global simple
.global done
simple:
    mov $1, %rdi
    add %rdi, %rax
done:
    ret

Overwriting /tmp/simple.s


In [10]:
! gcc -c /tmp/simple.s -o /tmp/simple.o

In [16]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode
ctx = pcode.BinaryContext("/tmp/simple.o")

Unexpected SP conflict


We can do fully automatic bounded model checking

In [18]:
init_mem = ctx.init_mem()
def prop_cb(addr, memstate):
    if addr == ctx.resolve_addr("done"):
        return memstate.get_reg(ctx, "RAX") == init_mem.get_reg(ctx, "RAX") + 2

ctx.model_check(init_mem, prop_cb, entries=["simple"], exits=["done"], insns=3)

0x400007
Counterexample found for property:  1 + select64le(register(state0), &RAX) ==
select64le(register(state0), &RAX) + 2
addr 0x400007


LemmaError: (Implies(And,
        1 +
        Concat(Concat(Concat(Concat(Concat(Concat(Concat(register(state0)[0 +
                                        8 -
                                        0 -
                                        1],
                                        register(state0)[0 +
                                        8 -
                                        1 -
                                        1]),
                                        register(state0)[0 +
                                        8 -
                                        2 -
                                        1]),
                                        register(state0)[0 +
                                        8 -
                                        3 -
                                        1]),
                                    register(state0)[0 + 8 -
                                      4 -
                                      1]),
                        

But to go further

Model requires 2 pieces of information
- How to extract the high level state from low level
- How the high level state evolves

In [19]:
def high_low(addr, memstate):
    i = memstate.get_reg(ctx, "RAX")
    return i

In [20]:
BV64 = BitVecSort(64)
@symdef
def step(i : BV64) -> BV64:
    return i + 1

Given this information, we can automatically check that the model and assembly operate in lockstep

In [21]:
ctx.bisim(high_low, step, entries=["simple"], exits=["done"])

Bisimulation proof succeeded  over all paths [('simple', 'done')]


Then we can prove properties about the model

In [22]:
from kdrag.all import *

i = Const("i", BV64)
@Theorem(ForAll([i], step(i) == i + 1))
def step_inc(l):
    i = l.fix()
    l.unfold(step)
    l.auto()
step_inc

|= ForAll(i, step(i) == i + 1)

# Clock

In [30]:
%%file /tmp/clock.s
    .global tick
    .global reset

reset:
    mov $1, %rdi
    jmp tick
tick:
    cmp $12, %rdi
    je reset
    add $1, %rdi
    jmp tick



Overwriting /tmp/clock.s


In [31]:
! gcc -c /tmp/clock.s -o /tmp/clock.o

In [32]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode

ctx = pcode.BinaryContext("/tmp/clock.o")

def high_low(addr, memstate):
    i = memstate.get_reg(ctx, "RDI")
    return i

BV64 = BitVecSort(64)
@symdef
def stepclk0(i : BV64) -> BV64:
    if i >= 12:
        return BitVecVal(1,64)
    else:
        return i + 1

ctx.bisim(high_low, stepclk0, cuts=["tick"])



Unexpected SP conflict


Counterexample found for bisimulation between high and low at 
 tick  --->  tick
Initial state:
 9223372036854775806 
steps to in low
 9223372036854775807 
steps to in high
 1


Exception: bisimulation failed

In [33]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode
ctx = pcode.BinaryContext("/tmp/clock.o")

def high_low(addr, memstate):
    i = memstate.get_reg(ctx, "RDI")
    return i

BV64 = smt.BitVecSort(64)
@symdef
def step(i : BV64) -> BV64:
    if i == 12:
        return BitVecVal(1,64)
    else:
        return i + 1

ctx.bisim(high_low, step, cuts=["tick"])

Unexpected SP conflict


Bisimulation proof succeeded  over all paths [('tick', 'tick')]


In [34]:
i = Const("i", BV64)
prove(ForAll([i], i != 0, ULE(i, 12), step(i) == stepclk0(i)), by=[stepclk0.defn, step.defn])

|= ForAll(i,
       Implies(And(i != 0, ULE(i, 12)),
               step(i) == stepclk0(i)))

In [36]:
hr = BitVec("hr", 64)

hr_inv = prove(ForAll([hr], hr != 0,  ULE(hr, 12), And(step(hr) != 0, ULE(step(hr),12))), by=[step.defn])
hr_inv

|= ForAll(hr,
       Implies(And(hr != 0, ULE(hr, 12)),
               And(step(hr) != 0, ULE(step(hr), 12))))

In [40]:
t = Int("t")
hr0 = BitVec("hr0", 64)
hr = kd.trans(step)(t, hr0)
@Theorem(ForAll([t, hr0],     hr0 != 0, ULE(hr0,12),
                          And(hr != 0,  ULE(hr, 12))))
def clock_inv(l):
    t, hr0 = l.fixes()
    l.induct(t)
    l.unfold(kd.trans(step))
    l.auto()

    l.unfold(kd.trans(step))
    l.auto()

    t = l.fix()
    l.intros()
    l.unfold(kd.trans(step))
    l.auto(by=[hr_inv])

clock_inv

|= ForAll([t, hr0],
       Implies(And(hr0 != 0, ULE(hr0, 12)),
               And(trans_step(t, hr0) != 0,
                   ULE(trans_step(t, hr0), 12))))

# Memcpy


In [41]:
%%file /tmp/mycpy.s
    .text
    .globl mycpy
    .align 2

# a0 is src
# a1 is dst
# a2 is len
mycpy:
    beq a2, x0, .done
.loop:
    lb t0, 0(a0)
    sb t0, 0(a1)
    addi a0, a0, 1
    addi a1, a1, 1
    addi a2, a2, -1
    bne a2, x0, .loop
.done:
    ret

Overwriting /tmp/mycpy.s


In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix | sh -s -- install --no-confirm

In [ ]:
! . /nix/var/nix/profiles/default/etc/profile.d/nix-daemon.sh && nix-shell -p pkgsCross.riscv32.buildPackages.gcc --run "riscv32-unknown-linux-gnu-gcc -c -O2 /tmp/mycpy.s  -o /tmp/mycpy"

In [42]:
! nix-shell -p pkgsCross.riscv32.buildPackages.gcc --run "riscv32-unknown-linux-gnu-gcc -c -O2 /tmp/mycpy.s  -o /tmp/mycpy"

In [1]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode

PC = kd.Enum("PC", ["mycpy", "loop", "done"])
BV32 = BitVecSort(32)
State = Struct("State", ("pc", PC), ("ram", smt.ArraySort(BV32, smt.BitVecSort(8))), ("len", BV32), ("src", BV32), ("dst", BV32))


In [6]:
labels = {
    "mycpy" : PC.mycpy,
    ".loop" : PC.loop,
    ".done" : PC.done
}
def high_low(addr, memstate):
    for label, pc in labels.items():
        if addr == ctx.resolve_addr(label):
            break
    else:
        raise Exception("Unexpected addr", hex(addr))
    return State(pc=pc, ram=memstate.ram, len=memstate.get_reg(ctx, "a2"), src=memstate.get_reg(ctx,"a0"), dst=memstate.get_reg(ctx,"a1"))



In [ ]:
@symdef
def step(st : State) -> State:
    if st.pc == PC.mycpy:
        if st.len == smt.BitVecVal(0, 32):
            st.pc = PC.done
        else:
            st.pc = PC.loop
    elif st.pc == PC.loop:
        st.ram = smt.Store(st.ram, st.dst, st.ram[st.src])
        st.len = st.len - 1
        st.src = st.src + 1
        st.dst = st.dst + 1
        if st.len == 0:
            st.pc = PC.done
        else:
            st.pc = PC.loop
    elif st.pc == PC.done:
        pass
    return st

In [7]:
ctx = pcode.BinaryContext("/tmp/mycpy", langid="RISCV:LE:32:default")
ctx.bisim(high_low, step, entries=["mycpy"], exits=[".done"], cuts=[".loop"])

Bisimulation proof succeeded  over all paths [('mycpy', '.loop'), ('.loop', '.loop'), ('.loop', '.done')]
